In [ ]:
!pip install -q pymupdf langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu streamlit

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

pdf_files = [file for file in os.listdir() if file.lower().endswith(".pdf")]

print("Total PDFs:", len(pdf_files))

for file in pdf_files:
    print(file)

Total PDFs: 4
venue.pdf
creta.pdf
Nexon Brochure.pdf
tata_punch_brochure_may_new.pdf


In [ ]:
brochure_metadata = {
    "venue.pdf": {
        "brand": "Hyundai",
        "model": "Venue"
    },
    "creta.pdf": {
        "brand": "Hyundai",
        "model": "Creta"
    },
    "Nexon Brochure.pdf": {
        "brand": "Tata",
        "model": "Nexon"
    },
    "tata_punch_brochure_may_new.pdf": {
        "brand": "Tata",
        "model": "Punch"
    }
}

for file, metadata in brochure_metadata.items():
    print(file, "->", metadata)

venue.pdf -> {'brand': 'Hyundai', 'model': 'Venue'}
creta.pdf -> {'brand': 'Hyundai', 'model': 'Creta'}
Nexon Brochure.pdf -> {'brand': 'Tata', 'model': 'Nexon'}
tata_punch_brochure_may_new.pdf -> {'brand': 'Tata', 'model': 'Punch'}


In [ ]:
import fitz

documents = []

for file, metadata in brochure_metadata.items():

    pdf = fitz.open(file)

    for page_number, page in enumerate(pdf):

        text = page.get_text()

        if text.strip():
            documents.append({
                "text": text,
                "brand": metadata["brand"],
                "model": metadata["model"],
                "page": page_number + 1,
                "source": file
            })

print("Total extracted pages:", len(documents))

Total extracted pages: 102


In [ ]:
print("Brand:", documents[0]["brand"])
print("Model:", documents[0]["model"])
print("Page:", documents[0]["page"])
print("Source:", documents[0]["source"])

print("\nExtracted Text:\n")
print(documents[0]["text"][:1000])

Brand: Hyundai
Model: Venue
Page: 1
Source: venue.pdf

Extracted Text:

VENUE



In [ ]:
from langchain_core.documents import Document

langchain_documents = []

for doc in documents:
    langchain_documents.append(
        Document(
            page_content=doc["text"],
            metadata={
                "brand": doc["brand"],
                "model": doc["model"],
                "page": doc["page"],
                "source": doc["source"]
            }
        )
    )

print("Total LangChain Documents:", len(langchain_documents))

Total LangChain Documents: 102


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(langchain_documents)

print("Total Chunks:", len(chunks))

Total Chunks: 175


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

/tmp/ipykernel_1635/2768797505.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_1635/2768797505.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS Vector Database created successfully!")

FAISS Vector Database created successfully!


In [ ]:
query = "What are the safety features of Creta?"

results = vector_store.similarity_search(
    query,
    k=5,
    filter={
        "brand": "Hyundai",
        "model": "Creta"
    }
)

print("Retrieved Chunks:", len(results))

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print("Brand:", doc.metadata["brand"])
    print("Model:", doc.metadata["model"])
    print("Page:", doc.metadata["page"])
    print("Source:", doc.metadata["source"])
    print("Text:", doc.page_content[:500])

Retrieved Chunks: 5

--- Result 1 ---
Brand: Hyundai
Model: Creta
Page: 8
Source: creta.pdf
Text: Protects what matters the most.
Hyundai CRETA comes equipped with 70+ advanced safety features including six airbags, as
standard. This SUV is a reassuringly secure sanctuary for its occupants.
Tyre pressure monitoring 
system: Highline
Electric parking brake with auto hold
Dashcam
Blind-spot view monitor (BVM)
Surround view monitor (SVM)
6 Airbags standard
Other features: ECM mirror with SOS button   I   ISOFIX   I   Front and rear parking sensors
36 Standard safety
features
6 airbags
ESC
VSM
H

--- Result 2 ---
Brand: Hyundai
Model: Creta
Page: 1
Source: creta.pdf
Text: CRETA

--- Result 3 ---
Brand: Hyundai
Model: Creta
Page: 9
Source: creta.pdf
Text: Adrenaline in DNA.
Powerfully distinctive design, impressive dimensions – Hyundai CRETA is an ultimate example of what 
happens when technological vision becomes a reality. It is as intrepid as it is agile. With the 1.5l petrol turbo 
unde

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Re-ranking model loaded successfully!")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Re-ranking model loaded successfully!


In [ ]:
pairs = [
    [query, doc.page_content]
    for doc in results
]

scores = reranker.predict(pairs)

reranked_results = sorted(
    zip(results, scores),
    key=lambda x: x[1],
    reverse=True
)

for i, (doc, score) in enumerate(reranked_results):
    print(f"\n--- Rank {i+1} ---")
    print("Score:", score)
    print("Page:", doc.metadata["page"])
    print("Text:", doc.page_content[:500])


--- Rank 1 ---
Score: 8.689211
Page: 8
Text: Protects what matters the most.
Hyundai CRETA comes equipped with 70+ advanced safety features including six airbags, as
standard. This SUV is a reassuringly secure sanctuary for its occupants.
Tyre pressure monitoring 
system: Highline
Electric parking brake with auto hold
Dashcam
Blind-spot view monitor (BVM)
Surround view monitor (SVM)
6 Airbags standard
Other features: ECM mirror with SOS button   I   ISOFIX   I   Front and rear parking sensors
36 Standard safety
features
6 airbags
ESC
VSM
H

--- Rank 2 ---
Score: 3.7124329
Page: 4
Text: Other features: Puddle lamp with welcome function   I   Chrome outside door handles   I  Front & rear skid plate
Integrated rear spoiler with LED HMSL (high mounted stop lamp)
What a legend looks like.
Bold. Charismatic. Extraordinary. Hyundai CRETA is symbolic of all things impressive no matter how you look 
at it. Its black chrome parametric grille makes a striking impression. Its horizon LED position

In [ ]:
top_k = 3

top_results = reranked_results[:top_k]

context = "\n\n".join([
    doc.page_content
    for doc, score in top_results
])

print("Selected Chunks:", len(top_results))
print("\nFinal Context:\n")
print(context)

Selected Chunks: 3

Final Context:

Protects what matters the most.
Hyundai CRETA comes equipped with 70+ advanced safety features including six airbags, as
standard. This SUV is a reassuringly secure sanctuary for its occupants.
Tyre pressure monitoring 
system: Highline
Electric parking brake with auto hold
Dashcam
Blind-spot view monitor (BVM)
Surround view monitor (SVM)
6 Airbags standard
Other features: ECM mirror with SOS button   I   ISOFIX   I   Front and rear parking sensors
36 Standard safety
features
6 airbags
ESC
VSM
HAC
All 
wheel 
disc 
brakes
New
New

Other features: Puddle lamp with welcome function   I   Chrome outside door handles   I  Front & rear skid plate
Integrated rear spoiler with LED HMSL (high mounted stop lamp)
What a legend looks like.
Bold. Charismatic. Extraordinary. Hyundai CRETA is symbolic of all things impressive no matter how you look 
at it. Its black chrome parametric grille makes a striking impression. Its horizon LED positioning lamp & DRLs 
and 

In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini API connected successfully!")

Gemini API connected successfully!


In [ ]:
prompt = f"""
You are DriveWise, an automotive assistant.

Answer the user's question only using the provided brochure context.
Do not use outside knowledge.
If the answer is not available in the context, say:
"Information not available in the selected brochure."

User Question:
{query}

Brochure Context:
{context}

Give a clear and concise answer.
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt
)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(response.text)

QUESTION:
What are the safety features of Creta?

ANSWER:
Hyundai CRETA is equipped with over 70 advanced safety features. Key safety features include:

*   **Airbags:** 6 airbags (standard)
*   **Monitoring Systems:** Tyre pressure monitoring system (Highline), Blind-spot view monitor (BVM), Surround view monitor (SVM), and Driver rear view monitor
*   **Braking and Stability:** Electric parking brake with auto hold, Electronic Stability Control (ESC), Vehicle Stability Management (VSM), Hill Assist Control (HAC), and All-wheel disc brakes
*   **Other Safety Features:** Dashcam, ECM mirror with SOS button, ISOFIX, and front and rear parking sensors.


In [ ]:
print("QUESTION:")
print(query)

print("\nANSWER:")
print(response.text)

print("\nSOURCES:")

seen_sources = set()

for doc, score in top_results:
    source_info = (
        doc.metadata["source"],
        doc.metadata["page"]
    )

    if source_info not in seen_sources:
        print(
            f"- {doc.metadata['source']} | Page {doc.metadata['page']}"
        )
        seen_sources.add(source_info)

QUESTION:
What are the safety features of Creta?

ANSWER:
Hyundai CRETA is equipped with over 70 advanced safety features. Key safety features include:

*   **Airbags:** 6 airbags (standard)
*   **Monitoring Systems:** Tyre pressure monitoring system (Highline), Blind-spot view monitor (BVM), Surround view monitor (SVM), and Driver rear view monitor
*   **Braking and Stability:** Electric parking brake with auto hold, Electronic Stability Control (ESC), Vehicle Stability Management (VSM), Hill Assist Control (HAC), and All-wheel disc brakes
*   **Other Safety Features:** Dashcam, ECM mirror with SOS button, ISOFIX, and front and rear parking sensors.

SOURCES:
- creta.pdf | Page 8
- creta.pdf | Page 4
- creta.pdf | Page 5


In [ ]:
def detect_section(text):
    text = text.lower()

    if "cng" in text and any(word in text for word in [
        "leak", "thermal", "fire protection", "gas"
    ]):
        return "cng_safety"

    elif any(word in text for word in [
        "adas", "lane keep", "collision warning",
        "emergency braking", "lane departure"
    ]):
        return "adas"

    elif any(word in text for word in [
        "airbag", "safety", "isofix", "stability",
        "tyre pressure", "blind view", "parking sensor",
        "fortified cabin"
    ]):
        return "safety"

    elif any(word in text for word in [
        "engine", "power", "torque", "transmission"
    ]):
        return "performance"

    elif any(word in text for word in [
        "infotainment", "speaker", "android auto",
        "apple carplay", "connectivity"
    ]):
        return "infotainment"

    elif any(word in text for word in [
        "seat", "comfort", "interior", "legroom"
    ]):
        return "interior"

    else:
        return "general"


for doc in langchain_documents:
    doc.metadata["section"] = detect_section(doc.page_content)

print("Section metadata added successfully!")

for doc in langchain_documents[:10]:
    print(
        doc.metadata["model"],
        "| Page:", doc.metadata["page"],
        "| Section:", doc.metadata["section"]
    )

Section metadata added successfully!
Venue | Page: 1 | Section: general
Venue | Page: 2 | Section: general
Venue | Page: 3 | Section: general
Venue | Page: 5 | Section: general
Venue | Page: 6 | Section: performance
Venue | Page: 7 | Section: general
Venue | Page: 8 | Section: interior
Venue | Page: 9 | Section: adas
Venue | Page: 10 | Section: general
Venue | Page: 11 | Section: adas


In [ ]:
# Recreate chunks with section metadata
chunks = text_splitter.split_documents(langchain_documents)

print("Total Updated Chunks:", len(chunks))

# Rebuild FAISS Vector Database
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS rebuilt with section metadata!")

Total Updated Chunks: 175
FAISS rebuilt with section metadata!


In [ ]:
for doc in langchain_documents:
    if doc.metadata["model"] == "Nexon":
        print(
            "Page:", doc.metadata["page"],
            "| Section:", doc.metadata["section"]
        )

Page: 1 | Section: safety
Page: 2 | Section: adas
Page: 3 | Section: general
Page: 4 | Section: general
Page: 5 | Section: general
Page: 6 | Section: interior
Page: 7 | Section: general
Page: 8 | Section: interior
Page: 9 | Section: interior
Page: 10 | Section: safety
Page: 11 | Section: safety
Page: 12 | Section: safety
Page: 15 | Section: adas
Page: 16 | Section: infotainment
Page: 17 | Section: performance
Page: 18 | Section: performance
Page: 19 | Section: general
Page: 20 | Section: interior
Page: 21 | Section: general
Page: 22 | Section: interior
Page: 23 | Section: general
Page: 24 | Section: general
Page: 25 | Section: general
Page: 26 | Section: performance
Page: 27 | Section: performance
Page: 28 | Section: general
Page: 29 | Section: cng_safety
Page: 31 | Section: interior
Page: 32 | Section: general
Page: 33 | Section: safety
Page: 35 | Section: adas
Page: 36 | Section: interior
Page: 37 | Section: safety
Page: 38 | Section: safety
Page: 39 | Section: adas
Page: 40 | Sectio

In [ ]:
def detect_section(text):
    text = text.lower()

    if "cng" in text and any(word in text for word in [
        "leak detection", "thermal incident",
        "fire protection", "gas leak"
    ]):
        return "cng_safety"

    elif any(word in text for word in [
        "lane keep assist",
        "front collision warning",
        "autonomous emergency braking",
        "lane departure warning",
        "traffic sign recognition"
    ]):
        return "adas"

    elif any(word in text for word in [
        "airbags",
        "isofix",
        "tyre pressure monitoring",
        "electronic stability",
        "blind view monitor",
        "surround view system",
        "fortified cabin",
        "parking sensors"
    ]):
        return "safety"

    elif any(word in text for word in [
        "engine", "power", "torque", "transmission"
    ]):
        return "performance"

    elif any(word in text for word in [
        "infotainment", "speaker",
        "android auto", "apple carplay"
    ]):
        return "infotainment"

    elif any(word in text for word in [
        "seat", "comfort", "interior", "legroom"
    ]):
        return "interior"

    return "general"


for doc in langchain_documents:
    doc.metadata["section"] = detect_section(doc.page_content)

print("Section detection improved!")

Section detection improved!


In [ ]:
def ask_drivewise(brand, model, question):

    # Detect query section
    section = detect_query_section(question)

    # Metadata filtered retrieval
    retrieved_docs = vector_store.similarity_search(
    question,
    k=10,
    fetch_k=200,
    filter={
        "brand": brand,
        "model": model,
        "section": section
    }
)

    # Fallback retrieval
    if not retrieved_docs:
        retrieved_docs = vector_store.similarity_search(
    question,
    k=10,
    fetch_k=200,
    filter={
        "brand": brand,
        "model": model
    }
)

    if not retrieved_docs:
        return {
            "answer": "Information not available in the selected brochure.",
            "sources": [],
            "section": section
        }

    # Re-ranking
    pairs = [
        [question, doc.page_content]
        for doc in retrieved_docs
    ]

    scores = reranker.predict(pairs)

    reranked_docs = sorted(
        zip(retrieved_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    # Context control with page diversity
    top_docs = []
    seen_pages = set()

    for doc, score in reranked_docs:
        page = doc.metadata["page"]

        if page not in seen_pages:
            top_docs.append((doc, score))
            seen_pages.add(page)

        if len(top_docs) == 3:
            break

    # Create final context
    context = "\n\n".join([
        doc.page_content
        for doc, score in top_docs
    ])

    # Prompt
    prompt = f"""
You are DriveWise, an automotive assistant.

Answer the user's question only using the provided brochure context.
Do not use outside knowledge.

If the answer is not available in the context, say:
"Information not available in the selected brochure."

User Question:
{question}

Brochure Context:
{context}

Give a clear and concise answer.
"""

    # Generate answer
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    # Sources
    sources = []

    for doc, score in top_docs:
        source = {
            "brochure": doc.metadata["source"],
            "page": doc.metadata["page"]
        }

        if source not in sources:
            sources.append(source)

    return {
        "answer": response.text,
        "sources": sources,
        "section": section
    }


print("Section-aware DriveWise RAG Pipeline Ready!")

Section-aware DriveWise RAG Pipeline Ready!


In [ ]:
chunks = text_splitter.split_documents(langchain_documents)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("Chunks and FAISS rebuilt successfully!")

Chunks and FAISS rebuilt successfully!


In [ ]:
result = ask_drivewise(
    brand="Tata",
    model="Nexon",
    question="What are the safety features of Nexon?"
)

print("DETECTED SECTION:")
print(result["section"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")

for source in result["sources"]:
    print(
        f"- {source['brochure']} | Page {source['page']}"
    )

DETECTED SECTION:
safety

ANSWER:
The safety features of the Nexon include:

*   **Airbags:** 6 Airbags (as standard).
*   **Cabin Structure:** Fortified cabin (an iron-clad skeleton system).
*   **Visibility & Monitoring:** Auto-dimming IRVM, Auto headlamps with rain-sensing wipers, Blind view monitor, 360° HD surround view system with front parking sensors, and HD rear view camera.
*   **Monitoring Systems:** Tyre pressure monitoring system.
*   **Emergency Assistance:** Emergency Call (E-Call) and Breakdown Call (B-Call).
*   **Seat & Restraint Systems:** 3-point ELR seatbelts (for all seats), ISOFIX, and all-adjustable headrests.
*   **Stability & Control:** Electronic Stability Program (as standard).
*   **Additional Features:** Rear defogger, central locking with key (and remote central locking flip key for Smart+), and follow-me-home headlamps.

SOURCES:
- Nexon Brochure.pdf | Page 11
- Nexon Brochure.pdf | Page 10
- Nexon Brochure.pdf | Page 38


In [ ]:
result = ask_drivewise(
    brand="Tata",
    model="Nexon",
    question="What are the safety features of Nexon?"
)

print("DETECTED SECTION:")
print(result["section"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")

for source in result["sources"]:
    print(
        f"- {source['brochure']} | Page {source['page']}"
    )

DETECTED SECTION:
safety

ANSWER:
The safety features of the Nexon include:

*   **Airbags:** 6 Airbags (as standard).
*   **Cabin Structure:** Fortified cabin (iron-clad skeleton system).
*   **Monitoring Systems:** Tyre Pressure Monitoring System, Blind View Monitor, and 360° HD Surround View System with front parking sensors.
*   **Visibility & Lighting:** Auto-dimming IRVM, Auto Headlamps with rain-sensing wipers, LED headlamps, DRLs, LED tail lamps, and Follow Me Home headlamps.
*   **Emergency Assistance:** Emergency Call (E-Call) and Breakdown Call (B-Call).
*   **General Safety:** 3-Point ELR for all seats, Rear Defogger, ISOFIX, Electronic Stability Program, and HD Rear View Camera.

SOURCES:
- Nexon Brochure.pdf | Page 11
- Nexon Brochure.pdf | Page 10
- Nexon Brochure.pdf | Page 38


In [ ]:
import pandas as pd
import time
import os
from datetime import datetime

print("Logging libraries loaded successfully!")

Logging libraries loaded successfully!


In [ ]:
LOG_FILE = "drivewise_logs.csv"

def save_log(brand, model, question, section, response_time, status):

    log_data = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "brand": brand,
        "model": model,
        "question": question,
        "section": section,
        "response_time": round(response_time, 2),
        "status": status
    }

    log_df = pd.DataFrame([log_data])

    if os.path.exists(LOG_FILE):
        log_df.to_csv(
            LOG_FILE,
            mode="a",
            header=False,
            index=False
        )
    else:
        log_df.to_csv(
            LOG_FILE,
            index=False
        )

    print("Query logged successfully!")


print("Logging function ready!")

Logging function ready!


In [ ]:
def ask_drivewise(brand, model, question):

    start_time = time.time()
    section = detect_query_section(question)

    try:
        # Metadata + section filtered retrieval
        retrieved_docs = vector_store.similarity_search(
            question,
            k=10,
            fetch_k=200,
            filter={
                "brand": brand,
                "model": model,
                "section": section
            }
        )

        # Fallback retrieval
        if not retrieved_docs:
            retrieved_docs = vector_store.similarity_search(
                question,
                k=10,
                fetch_k=200,
                filter={
                    "brand": brand,
                    "model": model
                }
            )

        if not retrieved_docs:
            response_time = time.time() - start_time

            save_log(
                brand,
                model,
                question,
                section,
                response_time,
                "Failed"
            )

            return {
                "answer": "Information not available in the selected brochure.",
                "sources": [],
                "section": section
            }

        # Re-ranking
        pairs = [
            [question, doc.page_content]
            for doc in retrieved_docs
        ]

        scores = reranker.predict(pairs)

        reranked_docs = sorted(
            zip(retrieved_docs, scores),
            key=lambda x: x[1],
            reverse=True
        )

        # Context control + page diversity
        top_docs = []
        seen_pages = set()

        for doc, score in reranked_docs:
            page = doc.metadata["page"]

            if page not in seen_pages:
                top_docs.append((doc, score))
                seen_pages.add(page)

            if len(top_docs) == 3:
                break

        # Create context
        context = "\n\n".join([
            doc.page_content
            for doc, score in top_docs
        ])

        # Prompt
        prompt = f"""
You are DriveWise, an automotive assistant.

Answer the user's question only using the provided brochure context.
Do not use outside knowledge.

If the answer is not available in the context, say:
"Information not available in the selected brochure."

User Question:
{question}

Brochure Context:
{context}

Give a clear and concise answer.
"""

        # Generate answer
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt
        )

        # Sources
        sources = []

        for doc, score in top_docs:
            source = {
                "brochure": doc.metadata["source"],
                "page": doc.metadata["page"]
            }

            if source not in sources:
                sources.append(source)

        response_time = time.time() - start_time

        # Save successful query log
        save_log(
            brand,
            model,
            question,
            section,
            response_time,
            "Success"
        )

        return {
            "answer": response.text,
            "sources": sources,
            "section": section
        }

    except Exception as e:

        response_time = time.time() - start_time

        save_log(
            brand,
            model,
            question,
            section,
            response_time,
            "Failed"
        )

        return {
            "answer": f"Error: {str(e)}",
            "sources": [],
            "section": section
        }


print("DriveWise RAG Pipeline with Logging Ready!")

DriveWise RAG Pipeline with Logging Ready!


In [ ]:
result = ask_drivewise(
    brand="Tata",
    model="Nexon",
    question="What are the safety features of Nexon?"
)

print("ANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(f"- {source['brochure']} | Page {source['page']}")

Query logged successfully!
ANSWER:
The safety features of the Nexon include:

*   **Airbags:** 6 Airbags (as standard).
*   **Cabin Structure:** Fortified cabin (iron-clad skeleton system).
*   **Monitoring & Visibility:** Tyre Pressure Monitoring System, Auto headlamps with rain-sensing wipers, Blind View Monitor, and a 360° HD Surround View System with front parking sensors.
*   **Assistance:** Emergency Call (E-Call) and Breakdown Call (B-Call) for 24/7 assistance.
*   **Additional Safety Features:** Auto-dimming IRVM, 3-Point ELR for all seats, Rear Defogger, ISOFIX, and Electronic Stability Program (as standard).
*   **Smart/Smart+ Trim Safety Features:** LED Head Lamps & DRLs, LED Tail Lamps, Central Locking, All-adjustable Headrest, and HD Rear View Camera.

SOURCES:
- Nexon Brochure.pdf | Page 11
- Nexon Brochure.pdf | Page 10
- Nexon Brochure.pdf | Page 38


In [ ]:
logs_df = pd.read_csv("drivewise_logs.csv")

logs_df

,timestamp,brand,model,question,section,response_time,status
0,2026-07-12 05:47:52,Tata,Nexon,What are the safety features of Nexon?,safety,2.05,Success


In [ ]:
vector_store.save_local("drivewise_faiss")

print("FAISS Vector Database saved successfully!")

FAISS Vector Database saved successfully!


In [ ]:
import os
import shutil

PROJECT_FOLDER = "drivewise_streamlit"

os.makedirs(PROJECT_FOLDER, exist_ok=True)

# Copy FAISS database
faiss_destination = os.path.join(
    PROJECT_FOLDER,
    "drivewise_faiss"
)

if os.path.exists(faiss_destination):
    shutil.rmtree(faiss_destination)

shutil.copytree(
    "drivewise_faiss",
    faiss_destination
)

print("Streamlit project folder created!")
print(os.listdir(PROJECT_FOLDER))

Streamlit project folder created!
['drivewise_faiss']


In [ ]:
app_code = '''
import streamlit as st
import time
import os
import pandas as pd

from datetime import datetime
from google import genai
from sentence_transformers import CrossEncoder
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


# Page Configuration
st.set_page_config(
    page_title="DriveWise",
    page_icon="🚗",
    layout="wide"
)


# Models
@st.cache_resource
def load_models():

    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vector_store = FAISS.load_local(
        "drivewise_faiss",
        embedding_model,
        allow_dangerous_deserialization=True
    )

    reranker = CrossEncoder(
        "cross-encoder/ms-marco-MiniLM-L-6-v2"
    )

    return vector_store, reranker


vector_store, reranker = load_models()


# Gemini Client
client = genai.Client(
    api_key=st.secrets["GEMINI_API_KEY"]
)


# Query Section Detection
def detect_query_section(question):

    question = question.lower()

    if "cng" in question:
        return "cng_safety"

    elif any(word in question for word in [
        "adas", "lane", "collision", "emergency braking"
    ]):
        return "adas"

    elif any(word in question for word in [
        "safety", "airbag", "isofix", "stability"
    ]):
        return "safety"

    elif any(word in question for word in [
        "engine", "power", "torque", "transmission"
    ]):
        return "performance"

    elif any(word in question for word in [
        "infotainment", "speaker",
        "android auto", "apple carplay"
    ]):
        return "infotainment"

    elif any(word in question for word in [
        "seat", "comfort", "interior", "legroom"
    ]):
        return "interior"

    return "general"


# Logging
LOG_FILE = "drivewise_logs.csv"


def save_log(
    brand,
    model,
    question,
    section,
    response_time,
    status
):

    log_data = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "brand": brand,
        "model": model,
        "question": question,
        "section": section,
        "response_time": round(response_time, 2),
        "status": status
    }

    log_df = pd.DataFrame([log_data])

    log_df.to_csv(
        LOG_FILE,
        mode="a",
        header=not os.path.exists(LOG_FILE),
        index=False
    )


# RAG Pipeline
def ask_drivewise(brand, model, question):

    start_time = time.time()

    section = detect_query_section(question)

    try:

        retrieved_docs = vector_store.similarity_search(
            question,
            k=10,
            fetch_k=200,
            filter={
                "brand": brand,
                "model": model,
                "section": section
            }
        )

        if not retrieved_docs:

            retrieved_docs = vector_store.similarity_search(
                question,
                k=10,
                fetch_k=200,
                filter={
                    "brand": brand,
                    "model": model
                }
            )

        if not retrieved_docs:

            response_time = time.time() - start_time

            save_log(
                brand,
                model,
                question,
                section,
                response_time,
                "Failed"
            )

            return {
                "answer": "Information not available in the selected brochure.",
                "sources": [],
                "section": section
            }


        # Re-ranking
        pairs = [
            [question, doc.page_content]
            for doc in retrieved_docs
        ]

        scores = reranker.predict(pairs)

        reranked_docs = sorted(
            zip(retrieved_docs, scores),
            key=lambda x: x[1],
            reverse=True
        )


        # Page Diversity
        top_docs = []
        seen_pages = set()

        for doc, score in reranked_docs:

            page = doc.metadata["page"]

            if page not in seen_pages:

                top_docs.append((doc, score))
                seen_pages.add(page)

            if len(top_docs) == 3:
                break


        context = "\\n\\n".join([
            doc.page_content
            for doc, score in top_docs
        ])


        prompt = f"""
You are DriveWise, an automotive assistant.

Answer the user's question only using the provided brochure context.

Do not use outside knowledge.

If the information is unavailable, say:
"Information not available in the selected brochure."

User Question:
{question}

Brochure Context:
{context}

Give a clear and concise answer.
"""


        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt
        )


        sources = []

        for doc, score in top_docs:

            source = {
                "brochure": doc.metadata["source"],
                "page": doc.metadata["page"]
            }

            if source not in sources:
                sources.append(source)


        response_time = time.time() - start_time

        save_log(
            brand,
            model,
            question,
            section,
            response_time,
            "Success"
        )


        return {
            "answer": response.text,
            "sources": sources,
            "section": section
        }


    except Exception as e:

        response_time = time.time() - start_time

        save_log(
            brand,
            model,
            question,
            section,
            response_time,
            "Failed"
        )

        return {
            "answer": f"Error: {str(e)}",
            "sources": [],
            "section": section
        }


# UI
st.title("🚗 DriveWise")

st.subheader(
    "Metadata-Aware Automotive RAG Assistant"
)

st.write(
    "Ask questions based on official car brochures."
)


car_data = {
    "Hyundai": ["Venue", "Creta"],
    "Tata": ["Nexon", "Punch"]
}


brand = st.selectbox(
    "Select Brand",
    list(car_data.keys())
)


model = st.selectbox(
    "Select Car Model",
    car_data[brand]
)


question = st.text_input(
    "Ask your question",
    placeholder="Example: What are the safety features?"
)


if st.button("Ask DriveWise"):

    if question.strip():

        with st.spinner(
            "Searching brochure..."
        ):

            result = ask_drivewise(
                brand,
                model,
                question
            )


        st.subheader("Answer")

        st.write(result["answer"])


        st.subheader("Sources")

        for source in result["sources"]:

            st.write(
                f"📄 {source['brochure']} | Page {source['page']}"
            )


        st.caption(
            f"Detected Section: {result['section']}"
        )

    else:

        st.warning(
            "Please enter a question."
        )
'''

with open(
    "drivewise_streamlit/app.py",
    "w",
    encoding="utf-8"
) as file:
    file.write(app_code)

print("app.py created successfully!")

app.py created successfully!


In [ ]:
requirements = """
streamlit
google-genai
sentence-transformers
langchain
langchain-community
langchain-text-splitters
faiss-cpu
pandas
"""

with open(
    "drivewise_streamlit/requirements.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(requirements.strip())

print("requirements.txt created successfully!")

requirements.txt created successfully!


In [ ]:
import os

for root, dirs, files in os.walk("drivewise_streamlit"):
    level = root.replace("drivewise_streamlit", "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

drivewise_streamlit/
    requirements.txt
    app.py
    drivewise_faiss/
        index.pkl
        index.faiss


In [ ]:
!streamlit run drivewise_streamlit/app.py --server.port 8501 &>/content/streamlit.log &

In [ ]:
!npx --yes localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴your url is: https://slick-pianos-kick.loca.lt
^C


In [ ]:
from google.colab import userdata
import os

api_key = userdata.get("GEMINI_API_KEY")

os.makedirs("/content/.streamlit", exist_ok=True)

with open("/content/.streamlit/secrets.toml", "w") as file:
    file.write(f'GEMINI_API_KEY = "{api_key}"\n')

print("Secret file exists:",
      os.path.exists("/content/.streamlit/secrets.toml"))

Secret file exists: True


In [ ]:
!pkill -f streamlit

In [ ]:
!streamlit run drivewise_streamlit/app.py --server.port 8501 &>/content/streamlit.log &

In [ ]:
!tail -20 /content/streamlit.log



2026-07-12 06:07:33.957 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.80.87.31:8501



In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("Cloudflared ready!")

Cloudflared ready!


In [ ]:
!./cloudflared tunnel --url http://localhost:8501

2026-07-12T06:09:53Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-12T06:09:53Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-12T06:09:57Z INF +--------------------------------------------------------------------------------------------+
2026-07-12T06:09:57Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-07-12T06:09:57Z INF |  https://lanka-thumbzilla-databases-jackie.trycloudfla

In [ ]:
gitignore = """
.streamlit/secrets.toml
drivewise_logs.csv
__pycache__/
*.pyc
"""

with open(
    "drivewise_streamlit/.gitignore",
    "w",
    encoding="utf-8"
) as file:
    file.write(gitignore.strip())

print(".gitignore created successfully!")

.gitignore created successfully!


In [ ]:
readme = """
# 🚗 DriveWise – Metadata-Aware Automotive RAG Assistant

DriveWise is a Retrieval-Augmented Generation (RAG) based automotive assistant that answers user questions using official car brochures.

## Features

- Brochure-based question answering
- Metadata filtering by brand and car model
- Section-aware retrieval
- FAISS vector database
- CrossEncoder re-ranking
- Context window control
- Source attribution with brochure page numbers
- Query logging and response time tracking
- Streamlit web interface

## Technologies Used

- Python
- LangChain
- FAISS
- Sentence Transformers
- CrossEncoder
- Gemini API
- Streamlit
- Pandas

## Supported Cars

- Hyundai Venue
- Hyundai Creta
- Tata Nexon
- Tata Punch

## RAG Workflow

PDF Brochures → Text Extraction → Chunking → Embeddings → FAISS → Metadata Filtering → Re-ranking → Context Control → Gemini → Answer + Sources

## Author

Udit Gupta
"""

with open(
    "drivewise_streamlit/README.md",
    "w",
    encoding="utf-8"
) as file:
    file.write(readme.strip())

print("README.md created successfully!")

README.md created successfully!


In [ ]:
import shutil

shutil.make_archive(
    "drivewise_final_project",
    "zip",
    "drivewise_streamlit"
)

print("Final project ZIP created!")

Final project ZIP created!


In [ ]:
from google.colab import files

files.download("drivewise_final_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>